# Training the food image classifier

Run in Google Colab with a GPU: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Takes about 20 minutes.

Dataset: [rajistics/indian_food_images](https://huggingface.co/datasets/rajistics/indian_food_images), 20 dishes, 5,328 training and 941 test images.

What this notebook does:
1. Trains a small CNN from scratch (baseline).
2. Fine-tunes EfficientNet-B0 pre-trained on ImageNet (transfer learning).
3. Compares them on the test set: accuracy, top-3 accuracy, macro F1, confusion matrix.
4. Exports the better model to ONNX so the web app can run it without PyTorch.

At the end it downloads `model_files.zip`. Unzip it into the repo's `models/` and `reports/` folders.

In [ ]:
!pip -q install datasets onnx onnxscript onnxruntime
import json, time, copy, numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
torch.manual_seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
ds = load_dataset("rajistics/indian_food_images")
LABELS = ds["train"].features["label"].names
train_idx, val_idx = train_test_split(np.arange(len(ds["train"])), test_size=0.1,
                                      stratify=ds["train"]["label"], random_state=42)
print(len(train_idx), "train,", len(val_idx), "validation,", len(ds["test"]), "test,", len(LABELS), "classes")
print(LABELS)

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
                               transforms.ColorJitter(0.2, 0.2, 0.2), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class Food(Dataset):
    def __init__(self, split, idx, tf):
        self.split, self.idx, self.tf = split, idx, tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        row = self.split[int(self.idx[i])]
        return self.tf(row["image"].convert("RGB")), row["label"]

loaders = {
    "train": DataLoader(Food(ds["train"], train_idx, train_tf), batch_size=48, shuffle=True, num_workers=2),
    "val": DataLoader(Food(ds["train"], val_idx, eval_tf), batch_size=96, num_workers=2),
    "test": DataLoader(Food(ds["test"], np.arange(len(ds["test"])), eval_tf), batch_size=96, num_workers=2),
}

In [ ]:
def run(model, epochs, lr, freeze_backbone_epochs=0):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_acc, best_state, history = 0, None, []
    for epoch in range(epochs):
        if hasattr(model, "features"):   # transfer learning: train only the head for the first epochs
            for p in model.features.parameters():
                p.requires_grad = epoch >= freeze_backbone_epochs
        model.train(); t0 = time.time()
        for x, y in loaders["train"]:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step()
        sched.step()
        y_true, y_pred, _ = predict(model, "val")
        acc = accuracy_score(y_true, y_pred); history.append(acc)
        print(f"epoch {epoch + 1}: val accuracy {acc:.3f} ({time.time() - t0:.0f}s)")
        if acc > best_acc:
            best_acc, best_state = acc, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, history

@torch.no_grad()
def predict(model, split):
    model.eval(); ys, ps, probs = [], [], []
    for x, y in loaders[split]:
        out = torch.softmax(model(x.to(device)), 1).cpu()
        ys += y.tolist(); ps += out.argmax(1).tolist(); probs.append(out)
    return np.array(ys), np.array(ps), torch.cat(probs).numpy()

def evaluate(model):
    y, p, prob = predict(model, "test")
    top3 = np.mean([yi in np.argsort(-pr)[:3] for yi, pr in zip(y, prob)])
    return {"accuracy": round(accuracy_score(y, p), 3), "top3_accuracy": round(float(top3), 3),
            "macro_f1": round(f1_score(y, p, average="macro"), 3)}, y, p

In [ ]:
# 1) Baseline: small CNN trained from scratch
def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
small_cnn = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256), block(256, 256),
                          nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, len(LABELS)))
small_cnn, hist_cnn = run(small_cnn, epochs=15, lr=2e-3)
res_cnn, _, _ = evaluate(small_cnn)
print("Small CNN:", res_cnn)

In [ ]:
# 2) Transfer learning: EfficientNet-B0 pre-trained on ImageNet
effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, len(LABELS))
effnet, hist_eff = run(effnet, epochs=10, lr=1e-3 / 3, freeze_backbone_epochs=2)
res_eff, y_test, p_test = evaluate(effnet)
print("EfficientNet-B0:", res_eff)
print(classification_report(y_test, p_test, target_names=LABELS, digits=3))

In [ ]:
import os
os.makedirs("out/models", exist_ok=True); os.makedirs("out/reports", exist_ok=True)

cm = confusion_matrix(y_test, p_test)
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(len(LABELS)), LABELS, rotation=90); ax.set_yticks(range(len(LABELS)), LABELS)
for (i, j), v in np.ndenumerate(cm):
    if v: ax.text(j, i, v, ha="center", va="center", fontsize=8, color="white" if v > cm.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("EfficientNet-B0, test set")
fig.tight_layout(); fig.savefig("out/reports/confusion_matrix.png", dpi=120)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(hist_cnn) + 1), hist_cnn, label="Small CNN (from scratch)")
ax.plot(range(1, len(hist_eff) + 1), hist_eff, label="EfficientNet-B0 (transfer)")
ax.set_xlabel("epoch"); ax.set_ylabel("validation accuracy"); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig("out/reports/training_curves.png", dpi=120)

per_class = classification_report(y_test, p_test, target_names=LABELS, output_dict=True)
metrics = {"test_size": int(len(y_test)), "labels": LABELS,
           "test_results": {"Small CNN (from scratch)": res_cnn, "EfficientNet-B0 (transfer learning)": res_eff},
           "chosen_model": "EfficientNet-B0 (transfer learning)",
           "per_class_f1": {k: round(per_class[k]["f1-score"], 3) for k in LABELS}}
json.dump(metrics, open("out/models/metrics.json", "w"), indent=2)
json.dump(LABELS, open("out/models/labels.json", "w"))
print(json.dumps(metrics["test_results"], indent=2))

In [ ]:
# Export to ONNX (the web app uses onnxruntime, no PyTorch needed)
effnet = effnet.cpu().eval()
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(effnet, dummy, "out/models/food_classifier.onnx", input_names=["image"], output_names=["logits"],
                  dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)

import onnxruntime as ort   # check the ONNX model gives the same answers as PyTorch
sess = ort.InferenceSession("out/models/food_classifier.onnx")
x = next(iter(loaders["test"]))[0][:8]
diff = np.abs(sess.run(None, {"image": x.numpy()})[0] - effnet(x).detach().numpy()).max()
print("max difference PyTorch vs ONNX:", diff)
print("size MB:", round(os.path.getsize("out/models/food_classifier.onnx") / 1e6, 1))

In [ ]:
!cd out && zip -qr ../model_files.zip models reports
from google.colab import files
files.download("model_files.zip")